# Phase 2: Data Cleaning & Feature Engineering

Mục tiêu của phase này:
- Làm sạch dữ liệu dựa trên kết quả của Phase 1.
- Chuẩn hóa tên cột.
- Tạo surrogate key `order_id`.
- Xử lý missing values và giá trị ngoại lệ (`not_defined`).
- Tạo các feature mới phục vụ phân tích.
- Xuất dữ liệu sạch ra file mới.

In [1]:
import pandas as pd
import numpy as np

# 1. Load raw data
df = pd.read_csv('../E-commerce Dataset.csv')
print(f"Raw data shape: {df.shape}")
df.head()

Raw data shape: (51290, 16)


,Order_Date,Time,Aging,Customer_Id,Gender,Device_Type,Customer_Login_type,Product_Category,Product,Sales,Quantity,Discount,Profit,Shipping_Cost,Order_Priority,Payment_method
0,1/2/2018,10:56:33,8.0,37077,Female,Web,Member,Auto & Accessories,Car Media Players,140.0,1.0,0.3,46.0,4.6,Medium,credit_card
1,7/24/2018,20:41:37,2.0,59173,Female,Web,Member,Auto & Accessories,Car Speakers,211.0,1.0,0.3,112.0,11.2,Medium,credit_card
2,11/8/2018,8:38:49,8.0,41066,Female,Web,Member,Auto & Accessories,Car Body Covers,117.0,5.0,0.1,31.2,3.1,Critical,credit_card
3,4/18/2018,19:28:06,7.0,50741,Female,Web,Member,Auto & Accessories,Car & Bike Care,118.0,1.0,0.3,26.2,2.6,High,credit_card
4,8/13/2018,21:18:39,9.0,53639,Female,Web,Member,Auto & Accessories,Tyre,250.0,1.0,0.3,160.0,16.0,Critical,credit_card


## 2. Chuẩn hóa tên cột (Standardize Column Names)
Chuyển tất cả tên cột về định dạng `snake_case`.

In [2]:
df.columns = df.columns.str.lower().str.replace(' ', '_')
print("Các cột sau khi chuẩn hóa:")
print(df.columns.tolist())

Các cột sau khi chuẩn hóa:
['order_date', 'time', 'aging', 'customer_id', 'gender', 'device_type', 'customer_login_type', 'product_category', 'product', 'sales', 'quantity', 'discount', 'profit', 'shipping_cost', 'order_priority', 'payment_method']


## 3. Tạo Surrogate Key (`order_id`)
Do dữ liệu gốc không có mã đơn hàng, ta sẽ tạo một `order_id` duy nhất cho mỗi dòng.

In [3]:
df.insert(0, 'order_id', ['ORD_' + str(i).zfill(6) for i in range(1, len(df) + 1)])
df[['order_id', 'order_date', 'customer_id']].head()

,order_id,order_date,customer_id
0,ORD_000001,1/2/2018,37077
1,ORD_000002,7/24/2018,59173
2,ORD_000003,11/8/2018,41066
3,ORD_000004,4/18/2018,50741
4,ORD_000005,8/13/2018,53639


## 4. Xử lý Missing Values & Data Types
Từ báo cáo Phase 1, ta thực hiện các bước làm sạch sau:
- `order_priority`: Điền 'Unknown' cho 2 giá trị thiếu.
- `payment_method`: Đổi 'not_defined' thành 'Unknown', chuẩn hóa format hiển thị.
- `order_date`: Ép kiểu về datetime.
- Các cột số: Ép kiểu numeric, điền missing bằng giá trị trung vị (median) của nhóm `product_category` tương ứng.

In [4]:
# 4.1 Xử lý Categorical missing/outlier
df['order_priority'] = df['order_priority'].fillna('Unknown')

df['payment_method'] = df['payment_method'].replace('not_defined', 'Unknown')
df['payment_method'] = df['payment_method'].str.replace('_', ' ').str.title()

# 4.2 Convert date
df['order_date'] = pd.to_datetime(df['order_date'], format='mixed', errors='coerce')

# 4.3 Xử lý Numeric missing
numeric_cols = ['sales', 'quantity', 'discount', 'profit', 'shipping_cost', 'aging']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    # Điền NA bằng median của nhóm danh mục sản phẩm
    df[col] = df.groupby('product_category')[col].transform(lambda x: x.fillna(x.median()))

print("Số lượng missing values sau xử lý:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Số lượng missing values sau xử lý:
Series([], dtype: int64)


## 5. Feature Engineering
Tạo các feature phục vụ việc phân tích BI:
- Kích thước thời gian: `order_month`, `order_quarter`, `order_weekday`
- Chỉ số tỷ lệ: `profit_margin`, `shipping_cost_ratio`

In [5]:
# Time features
df['order_month'] = df['order_date'].dt.to_period('M').astype(str)
df['order_quarter'] = 'Q' + df['order_date'].dt.quarter.astype(str) + '-' + df['order_date'].dt.year.astype(str)
df['order_weekday'] = df['order_date'].dt.day_name()

# Ratio features
df['profit_margin'] = np.where(df['sales'] > 0, df['profit'] / df['sales'], 0).round(4)
df['shipping_cost_ratio'] = np.where(df['sales'] > 0, df['shipping_cost'] / df['sales'], 0).round(4)

df[['sales', 'profit', 'profit_margin', 'shipping_cost', 'shipping_cost_ratio', 'order_month', 'order_quarter']].head()

,sales,profit,profit_margin,shipping_cost,shipping_cost_ratio,order_month,order_quarter
0,140.0,46.0,0.3286,4.6,0.0329,2018-01,Q1-2018
1,211.0,112.0,0.5308,11.2,0.0531,2018-07,Q3-2018
2,117.0,31.2,0.2667,3.1,0.0265,2018-11,Q4-2018
3,118.0,26.2,0.2220,2.6,0.0220,2018-04,Q2-2018
4,250.0,160.0,0.6400,16.0,0.0640,2018-08,Q3-2018


## 6. Export Clean Data
Kiểm tra chất lượng dữ liệu lần cuối và lưu thành file CSV mới.

In [6]:
import os

df.info()

os.makedirs('../data', exist_ok=True)
clean_path = '../data/cleaned_ecommerce_dataset.csv'
df.to_csv(clean_path, index=False)

print(f"\nData đã được làm sạch và lưu tại: {clean_path}")
print(f"Final shape: {df.shape}")

<class 'pandas.DataFrame'>
RangeIndex: 51290 entries, 0 to 51289
Data columns (total 22 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   order_id             51290 non-null  str           
 1   order_date           51290 non-null  datetime64[us]
 2   time                 51290 non-null  str           
 3   aging                51290 non-null  float64       
 4   customer_id          51290 non-null  int64         
 5   gender               51290 non-null  str           
 6   device_type          51290 non-null  str           
 7   customer_login_type  51290 non-null  str           
 8   product_category     51290 non-null  str           
 9   product              51290 non-null  str           
 10  sales                51290 non-null  float64       
 11  quantity             51290 non-null  float64       
 12  discount             51290 non-null  float64       
 13  profit               51290 non-null  float


Data đã được làm sạch và lưu tại: ../data/cleaned_ecommerce_dataset.csv
Final shape: (51290, 22)
